# acquire

> pull the web, papers, video, files, code and JSON APIs into the vault — once, or on a schedule

In [ ]:
#| default_exp acquire

In [ ]:
#| hide
from nbdev.showdoc import *
from fastcore.test import *

Every method here is one [fossick](https://github.com/vedicreader/fossick) call plus `Vault.add`.
fossick knows how to get past bot walls, read arXiv and YouTube, and sniff a page's JSON API; the
vault's job is only to file what comes back with the provenance that explains why it is there.

In [ ]:
#| export
import json, re, time, uuid, warnings
from urllib.parse import urlparse
from fastcore.all import AttrDict, L, Path, patch
from litesearch import code_exts, dir2files, pdf_parse, DOC_EXTS
from vishalakshi.core import Vault, KINDS

In [ ]:
#| export
def clip(s:str, n:int=120) -> str:
    'Collapse whitespace and clip a scraped title to something a breadcrumb can carry.'
    return re.sub(r'\s+', ' ', (s or '').strip())[:n] or 'untitled'

def md_title(md:str, fallback:str='') -> str:
    "First markdown heading in `md`, else `fallback` — scraped <title>s are often junk."
    m = re.search(r'^#{1,2} +(.+)$', md or '', flags=re.M)
    return clip(m.group(1) if m else fallback)

In [ ]:
#| export
@patch
def url(self:Vault,
        url:str,            # page to read
        title:str=None,     # defaults to the page's first heading, else its path
        sel:str=None,       # CSS selector to narrow the page before conversion
        kind:str='web',
        auto:bool=True,     # escalate plain -> heavy -> stealthy -> logged-in Chrome past bot walls
        meta:dict=None,
        force:bool=False,
        verify=False,
        **kw                # forwarded to fossick.fetch
) -> dict:
    """Fetch one URL, convert it to markdown and file it in the vault.

    `auto=True` is the default because a bot wall returns HTTP 200 with a challenge page, which
    would otherwise be indexed as if it were the article."""
    from fossick import fetch, to_md
    pg = fetch(url, sel=sel, auto=auto, verify=verify, **kw)
    md, st = (to_md(pg, sel=sel) if pg is not None else ''), getattr(pg, 'status', None)
    if not md.strip() or (st or 200) >= 400:            # a failed fetch is None, and a bot wall is a 4xx
        return dict(url=url, skipped=f'could not read the page (status {st})', status=st)
    m = dict(meta or {}, url=url, status=st, fetched_at=time.time())
    return dict(self.add(md, title or md_title(md, urlparse(url).path.rsplit('/', 1)[-1] or url),
                         source=url, kind=kind, meta=m, force=force), url=url)

@patch
def crawl(self:Vault, start_url:str, max_pages:int=10, sel:str=None, verify=False, **kw) -> L:
    'Crawl a docs site or blog from a start URL and file every page in the vault.'
    from fossick import crawl as _crawl, to_md
    pgs = L((pg.url, to_md(pg, sel=sel)) for pg in _crawl(start_url, sel=sel, max_pages=max_pages, verify=verify,**kw))
    return L(self.add(md, md_title(md, u), source=u, kind='web',
                      meta=dict(url=u, via='crawl', root=start_url, fetched_at=time.time()))
             for u, md in pgs if md.strip())

@patch
def web(self:Vault,
        query:str,          # what to search for
        n:int=5,            # top results to read
        google:bool=False,  # real Google ranking via a stealth browser (slower)
        chars:int=60000,    # max markdown chars kept per source
        verify=False,
        **kw                # forwarded to fossick.research
) -> AttrDict:
    """Search the web, read the top `n` results, and file all of them in the vault.

    This is the loop the vault exists for: the query that found a page is kept in its metadata, so
    months later `sources()` still says *why* a document is in your corpus. Results already present
    are skipped rather than duplicated, so re-running an overlapping search is cheap."""
    from fossick import research
    res = research(query, n=n, engine='google' if google else 'search', chars=chars, verify=verify, **kw)
    srcs = L(res['sources']).filter(lambda s: s['md'].strip() and s['href'])
    added = srcs.map(lambda s: dict(self.add(s['md'], clip(s['title']), source=s['href'], kind='web',
                                             meta=dict(url=s['href'], query=query, fetched_at=time.time())),
                                    url=s['href']))
    return AttrDict(query=query, n_found=len(res['sources']), added=added)

In [ ]:
#| export
@patch
def arxiv(self:Vault, id_or_url:str, save_dir:str=None, force:bool=False, verify=False, **kw) -> dict:
    """Read an arXiv paper (metadata + full text) into the vault as `kind="arxiv"`.

    Files itself on the shelf `KIND_SHELF` names for a paper. A method whose *name* fixes the kind
    can route on its own; `pdf` and `add_file` cannot, because there the argument decides."""
    if (v := self.route('arxiv')) is not self:
        return v.arxiv(id_or_url, save_dir=save_dir, force=force, verify=verify, **kw)
    from fossick import read_arxiv
    p = read_arxiv(id_or_url, save_dir=save_dir or str(self.assets('pdfs')), force=force, verify=verify, **kw)
    md = f"# {p['title']}\n\n{p.get('summary','')}\n\n{p.get('source') or ''}"
    return self.add(md, clip(p['title']), source=p.get('link') or id_or_url, kind='arxiv', force=force,
                    meta=dict(authors=list(p.get('authors') or []), published=p.get('published'),
                              pdf_path=p.get('pdf_path'), fetched_at=time.time()))

@patch
def pdf(self:Vault, path_or_url:str, title:str=None, force:bool=False, verify=False, **kw) -> dict:
    'Read a PDF (local path or URL) into the vault, one tree node per heading.'
    from fossick import get_pdf
    p = Path(path_or_url)
    if p.exists(): return self.add_file(p, title=title, kind='pdf', force=force)
    doc = get_pdf(path_or_url, verify=verify, **kw)
    if doc is None: return dict(source=path_or_url, skipped='not a PDF or could not be fetched')
    stem = path_or_url.rsplit('/', 1)[-1].split('?')[0]
    return self.add(list(enumerate(pdf_parse(doc, out_path=self.assets(stem or 'pdf')))),
                    title or clip(stem), source=path_or_url, kind='pdf', force=force,
                    meta=dict(url=path_or_url, fetched_at=time.time()))

@patch
def youtube(self:Vault, url:str, force:bool=False) -> dict:
    "Read a YouTube video's transcript and metadata into the vault."
    from fossick import read_yt
    v = read_yt(url, force=force)
    if not (v.get('source') or '').strip():
        return dict(source=url, skipped='no transcript available', title=v.get('title'))
    md = f"# {v['title']}\n\n{v.get('description','')}\n\n## Transcript\n\n{v['source']}"
    return self.add(md, clip(v['title']), source=url, kind='youtube', force=force,
                    meta=dict(url=url, channel=v.get('channel'), duration=v.get('duration'),
                              upload_date=v.get('upload_date'), fetched_at=time.time()))

def what_is(target:str) -> str:
    "Which kind of thing a `grab` target names: `dir`, `file`, `arxiv`, `youtube`, `pdf` or `web`."
    p = Path(target)
    if p.is_dir(): return 'dir'
    if p.exists(): return 'file'
    if 'arxiv.org' in target or re.fullmatch(r'\d{4}\.\d{4,5}(v\d+)?', target): return 'arxiv'
    if re.search(r'youtube\.com|youtu\.be', target): return 'youtube'
    if target.lower().split('?')[0].endswith('.pdf'): return 'pdf'
    if target.startswith('http'): return 'web'
    raise ValueError(f'not a URL, an arXiv id, a file or a directory: {target}')

@patch
def grab(self:Vault,
         target:str,       # a URL, an arXiv id, a YouTube link, a PDF, a local file or a directory
         title:str=None,
         sel:str=None,     # CSS selector, for the web cases
         shelf:str=None,   # shelf to file it on; None -> whichever `KIND_SHELF` names for its kind
         **kw              # forwarded to whichever method the target names
):
    """File anything, by looking at what it is — the one call a CLI or an agent needs.

    What it is decides two things: which method reads it, and which shelf it lands on. A paper goes
    to the shelf a science encoder wrote, a directory splits between the vault and kosha through
    `add_tree`, and everything unrouted stays here. `shelf=` overrides the route."""
    kind = what_is(target)
    v = self.shelf(shelf) if shelf else self.route(kind)
    if kind == 'dir':      return v.add_tree(target, **kw)
    if kind == 'file':     return v.add_file(target, title=title, **kw)
    if kind == 'arxiv':    return v.arxiv(target, **kw)
    if kind == 'youtube':  return v.youtube(target, **kw)
    if kind == 'pdf':      return v.pdf(target, title=title, **kw)
    return v.url(target, title=title, sel=sel, **kw)

@patch
def code(self:Vault, dir:str, types:str=code_exts, **kw) -> L:
    """File a source tree into the vault as `kind='code'`, so code and prose answer one query.
    This is deliberately the shallow path: files as documents, headings from the text. For call
    graphs, PageRank over symbols and `where_to_add`, use `index_code` — kosha builds an
    AST-derived index that this cannot, and `federate` searches both. `add_tree` does both halves of
    a mixed tree at once and is usually what you want."""
    return self.add_dir(dir, types=types, kind='code', **kw)

One directory, two indexes. `add_dir` files documents and `index_code` fills kosha; a tree usually
holds both, and remembering to call each is exactly the sort of thing a library should do for you.

In [ ]:
#| export
@patch
def add_tree(self:Vault,
             dir:str,                # tree to ingest
             types:str=DOC_EXTS,     # extensions filed into the vault as prose
             code:bool=True,         # index source files with kosha, when the tree has any
             kind:str=None,          # override the kind for the prose half
             connect:bool=True,      # rebuild the entity graph once, at the end
             verbose:bool=False,
             **kw                    # forwarded to add_file
) -> AttrDict:
    """Ingest a whole tree, each half to the index that can actually answer questions about it.

    A mixed tree is the normal case — a repo with a README, docs, notebooks and source — and prose
    and code want different indexes. Prose wants chunks, headings and embeddings. Code wants an AST:
    `symbol()`, `where_to_add()` and the call graph exist only in kosha, and filing a `.py` file into
    the prose store as a document (which is what `Vault.code()` does) buys none of them. So
    `add_dir` handles the documents, `index_code` handles the source, and this is the one call that
    knows which is which.

    The graph is rebuilt once at the end rather than per file, for the reason `poll` rebuilds it once
    per tick: `connect()` reads the whole store, so a tree of two hundred files would otherwise pay
    for it two hundred times. If kosha is not installed, the source files are filed as prose instead
    and `code` says so — a searchable fallback beats a traceback.

    `code` is a plain switch rather than a three-way choice because forcing kosha at a tree with no
    source in it would index nothing at the price of loading a code embedder."""
    p = Path(dir)
    if not p.is_dir(): raise ValueError(f'not a directory: {dir}')
    docs = self.add_dir(p, types=types, kind=kind, **kw)
    srcs = dir2files(p, types=code_exts) if code else L()
    out = AttrDict(dir=str(p), docs=docs, n_docs=len(docs), n_code=len(srcs), code=None)
    if srcs:
        try: out.code = self.index_code(p, verbose=verbose)
        except Exception as e:
            warnings.warn(f'could not index {len(srcs)} source files with kosha '
                          f'({type(e).__name__}: {str(e)[:120]}) — filing them as prose instead, so '
                          f'they are at least searchable. Install kosha for symbol search.')
            out.code = dict(error=f'{type(e).__name__}: {str(e)[:200]}', filed_as_prose=len(srcs))
            out.docs = docs + srcs.map(self.add_file, kind='code', **kw)
    if connect and (out.n_docs or out.code): out.graph = self.connect()
    return out

### Harvest: read a page's API, not its HTML

Listing, product and dashboard pages render from an internal JSON API. Reading that API is faster,
paginates cleanly and survives redesigns, where scraping the DOM does none of those. One document
per harvest, one `##` section per record — so `build_tree` gives every record its own node and
breadcrumb, and a catalogue becomes individually retrievable rows sitting next to your notes.

In [ ]:
#| export
def records(data, min_len:int=2) -> list:
    """The longest list of dicts inside an arbitrary JSON response.
    APIs bury their payload at different depths (`results`, `data.products.items`, a bare array),
    so this walks for the longest list of dicts rather than guessing a key name."""
    best = []
    def walk(o, d=0):
        nonlocal best
        if d > 6: return
        if isinstance(o, list):
            ds = [x for x in o if isinstance(x, dict)]
            if len(ds) >= min_len and len(ds) > len(best): best = ds
            for x in o[:20]: walk(x, d+1)
        elif isinstance(o, dict):
            for v in o.values(): walk(v, d+1)
    walk(data)
    return best

def records_md(recs, title_keys=('name', 'title', 'displayName', 'productName', 'label', 'sku', 'id')) -> str:
    'Records as markdown, one `##` section per record, so each becomes its own retrievable node.'
    def ttl(r): return clip(next((r[k] for k in title_keys if isinstance(r.get(k), str) and r[k].strip()),
                                 json.dumps(r, default=str)), 80)
    return '\n\n'.join(f'## {ttl(r)}\n\n```json\n{json.dumps(r, indent=1, default=str)}\n```'
                       for r in L(recs).map(lambda r: r if isinstance(r, dict) else dict(value=r)))

In [ ]:
#| export
@patch
def apis(self:Vault,
         url:str,             # page to watch
         pattern:str='*',     # glob/regex filtering captured request URLs
         session:bool=False,  # capture through the logged-in debug Chrome
         preview:int=240,     # chars of each response shown
         **kw                 # forwarded to fossick.find_xhr
) -> L:
    'Discover the JSON endpoints a page calls, so you can read its data instead of its HTML.'
    from fossick import find_xhr
    self._caps = L(find_xhr(url, pattern=pattern, session=session, **kw))
    return L(AttrDict(n=i, url=h['url'], content_type=h.get('content_type'),
                      records=len(records(h.get('data'))),
                      preview=json.dumps(h.get('data'), default=str)[:preview])
             for i, h in enumerate(self._caps))

@patch
def harvest(self:Vault,
            url:str,                # the page whose API you want
            pattern:str='*',        # which captured request URLs to keep
            title:str=None,         # document title; defaults to the page host + path
            capture:int=None,       # replay a specific endpoint from the last apis() call
            pages:int=1,            # pages to pull; >1 paginates the endpoint
            page_field:str='page',  # query/body key incremented per page
            session:bool=False,     # capture through the logged-in Chrome
            force:bool=False,
            **kw
) -> dict:
    """Sniff a page's JSON API, pull the records, and file them in the vault as `kind='data'`.
    Re-harvesting the same source replaces rather than duplicates when `force=True`; otherwise it
    is a no-op, which is what makes this safe to put behind a `watch`."""
    from fossick import replay_xhr, paginate_api
    caps = getattr(self, '_caps', None)
    if capture is None or not caps:
        found = self.apis(url, pattern=pattern, session=session)
        if not found: return dict(url=url, skipped='no JSON endpoints captured')
        capture, caps = max(found, key=lambda h: h.records).n, self._caps
    cap = caps[capture].get('capture')
    if cap is None: return dict(url=url, skipped=f'capture {capture} is not replayable')
    ep = cap['url']
    if pages > 1: items = paginate_api(ep, payload=cap.get('request_body'), page_field=page_field,
                                       method=(cap.get('method') or 'GET').upper(), max_pages=pages, **kw)
    else:
        try: items = records(replay_xhr(cap, **kw).json())
        except Exception as e: return dict(url=url, endpoint=ep, skipped=f'{type(e).__name__}: {e}')
    if not items: return dict(url=url, endpoint=ep, skipped='no records found in the response')
    ttl = title or f'{urlparse(url).netloc}{urlparse(url).path}'.strip('/')
    return dict(self.add_records(items, ttl, source=ep, force=force, meta=dict(page=url, endpoint=ep,
           harvested_at=time.time())), endpoint=ep, records=len(items))

@patch
def add_records(self:Vault, recs:list, title:str, source:str=None, kind:str='data', force:bool=False,
                meta:dict=None) -> dict:
    'File a list of dicts you already have (any API, any export) as one document, one section each.'
    return self.add(f'# {title}\n\n{records_md(recs)}', title, source=source or f'records:{title}',
                    kind=kind, force=force, meta=dict(meta or {}, records=len(L(recs))))

### Watches: keeping it current

A watch is what turns the vault from an archive into something that stays current. `action` names
an acquisition method, so anything you can file once you can file on a schedule; `remind` writes a
note instead of fetching, which is the recurring-reminder case with no network involved. `poll()`
is the tick — cron, a scheduler, or a frontend button.

In [ ]:
#| export
ACTIONS = ('url', 'web', 'harvest', 'arxiv', 'youtube', 'crawl', 'remind')

def secs(every) -> float:
    "Seconds from `'30m'`, `'6h'`, `'2 days'`, `'1w'`, or a number of seconds."
    import pandas as pd
    return float(every) if isinstance(every, (int, float)) else pd.Timedelta(every).total_seconds()

@patch
def _w(self:Vault):
    'The watches table, created on first use.'
    t = self.db.t.watches
    t.create(id=str, action=str, target=str, params=str, every=float, note=str, enabled=int,
             last_run=float, last_status=str, next_run=float, runs=int, pk='id', if_not_exists=True)
    return t

@patch
def watch(self:Vault,
          target:str,         # URL, query, arXiv id, or the text of a reminder
          action:str='url',   # one of ACTIONS — what to do when it fires
          every:str='1d',     # interval: '30m', '6h', '1d', '1w', or seconds
          note:str=None,      # why you are watching
          start:float=None,   # first run time (epoch); defaults to now
          **params            # forwarded to the action (n=, pattern=, pages=, sel=, ...)
) -> dict:
    'Register a recurring job: re-read a page, re-run a search, re-harvest an API, or remind you.'
    assert action in ACTIONS, f'action must be one of {ACTIONS}'
    row = dict(id=uuid.uuid4().hex[:12], action=action, target=target, params=json.dumps(params),
               every=secs(every), note=note or '', enabled=1, next_run=start or time.time(), runs=0)
    self._w().insert(row, replace=True)
    return dict(row, params=params)

@patch
def watches(self:Vault, due_only:bool=False, at:float=None) -> L:
    'Every registered watch, soonest first; `due_only` keeps the ones whose next run has arrived.'
    where = f'enabled=1 AND next_run<={at or time.time()}' if due_only else None
    return L(self._w()(where=where, order_by='next_run')).map(
        lambda r: dict(r, params=json.loads(r['params'] or '{}')))

@patch
def unwatch(self:Vault, watch_id:str):
    'Delete a watch. The documents it already filed stay in the vault.'
    self._w().delete(watch_id)

@patch
def pause(self:Vault, watch_id:str, enabled:bool=False):
    'Disable (or re-enable) a watch without losing it.'
    self._w().update(dict(id=watch_id, enabled=int(enabled)))

@patch
def run_watch(self:Vault, w:dict) -> dict:
    """Fire one watch and record the outcome.

    A failure is recorded on the row and returned, never raised: one dead URL must not stop a
    polling loop from servicing every other watch."""
    t0 = time.time()
    try:
        res = (self.note(w['target'], title=w.get('note') or None, tags=['reminder'])
               if w['action'] == 'remind' else getattr(self, w['action'])(w['target'], **w['params']))
        status = 'skipped' if isinstance(res, dict) and res.get('skipped') else 'ok'
    except Exception as e: res, status = dict(error=f'{type(e).__name__}: {str(e)[:200]}'), 'error'
    now = time.time()
    self._w().update(dict(id=w['id'], last_run=now, last_status=status, runs=w['runs']+1,
                          next_run=now + w['every']))
    return dict(watch_id=w['id'], action=w['action'], target=w['target'], status=status,
                took=round(now-t0, 2), result=res)

@patch
def poll(self:Vault, at:float=None, limit:int=None, connect:bool=True) -> dict:
    """Run every watch that is due. This is the tick a scheduler, a cron or a frontend calls.

    Rebuilds the entity graph once at the end rather than per watch, because `connect()` reads the
    whole store and a poll that fired five watches would otherwise pay for it five times."""
    ran = self.watches(due_only=True, at=at)[:limit].map(self.run_watch)
    if connect and ran.filter(lambda r: r['status'] == 'ok'): self.connect()
    pending = self.watches()
    return dict(checked=len(pending), ran=len(ran), results=ran,
                next_due=pending[0]['next_run'] if pending else None)

## Try it

In [ ]:
v = Vault(':memory:')
v.add_records([dict(sku='A1', name='Free range eggs', price=4.5),
               dict(sku='B2', name='Oat milk', price=2.1)], 'dairy')
v.find('eggs')[0]['breadcrumb']

/Users/71293/code/personal/orgs/vishalakshi/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'dairy › Free range eggs'

In [ ]:
test_eq(len(records({'data': {'items': [{'a': 1}, {'a': 2}, {'a': 3}]}})), 3)
test_eq(secs('6h'), 21600); test_eq(secs('1w'), 604800); test_eq(secs(90), 90)
w = v.watch('late chunking', action='web', every='1d', n=3)
test_eq(w['params'], dict(n=3))
test_eq(len(v.watches(due_only=True)), 1)
v.unwatch(w['id'])
test_eq(len(v.watches()), 0)

/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_76490/529824922.py:7: Pandas4Warning: 'w' is deprecated and will be removed in a future version. Please use 'W' instead of 'w'.
  return float(every) if isinstance(every, (int, float)) else pd.Timedelta(every).total_seconds()
/var/folders/kg/9vdw4mdd1fs58svgh4k1qhr09x7dqh/T/ipykernel_76490/529824922.py:7: Pandas4Warning: 'd' is deprecated and will be removed in a future version. Please use 'D' instead of 'd'.
  return float(every) if isinstance(every, (int, float)) else pd.Timedelta(every).total_seconds()


In [ ]:
#| hide
# a mixed tree — a README and a source file, the shape of any repo
from tempfile import mkdtemp
d = Path(mkdtemp())
(d/'README.md').write_text('# fuse\n\nRanks are fused because the legs share no vector space.')
(d/'fuse.py').write_text('def fuse(ranks):\n    "Reciprocal rank fusion over ranked lists."\n    return ranks\n')

r = Vault(':memory:').add_tree(d, connect=False)
test_eq((r.n_docs, r.n_code), (1, 1))                 # the doc to the vault, the source to kosha
test_eq(r.docs.attrgot('title'), ['README'])
assert r.code and not r.code.get('error'), r.code     # kosha answered, not the prose fallback

# code=False leaves the source where it is, and a tree with no source never reaches kosha at all
r = Vault(':memory:').add_tree(d, code=False, connect=False)
test_eq((r.n_code, r.code), (0, None))
test_fail(lambda: Vault(':memory:').add_tree(d/'README.md'), contains='not a directory')

# the graph is rebuilt once at the end, not once per file
r = Vault(':memory:').add_tree(d, connect=True)
assert r.graph['entities'] > 0, r.graph

In [ ]:
#| hide
# `arxiv` routes itself, because the *method* fixes the kind and `grab` would route it anyway
v2 = Vault(':memory:')
test_eq(v2.route('arxiv').store, 'papers')
# ...and nothing else does. `find` is a single-shelf primitive, so a write that quietly lands on
# another shelf is a read that quietly returns nothing — `code`'s whole purpose is that code and
# prose answer one query, and `add_records` takes its kind as an argument rather than in its name.
v2.add_records([dict(sku='A1', name='free range eggs')], 'groceries')
test_eq(v2.doc('records:groceries')['title'], 'groceries')       # right here, where `find` looks
assert v2.find('free range eggs')
(d/'b.py').write_text('def fuse(x):\n    return x\n')
v2.code(d)
assert 'b' in v2.sources(kind='code').attrgot('title')   # filed here, as `kind='code'` prose
# an explicit shelf is never overruled — what was asked for is what happens
test_eq(v2.shelf('sanskrit', offline=True).route('arxiv').store, 'sanskrit')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()